# TravelMate AI — Giai đoạn 2: Huấn luyện QLoRA

Notebook dùng Qwen3-4B ở chế độ 4-bit và chỉ huấn luyện LoRA adapter. Hãy chọn GPU runtime trong Colab trước khi chạy.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/trongnd16092005/travelmate-ai.git"
BRANCH = "feature/ai-itinerary-generation"
REPO_DIR = Path("/content/travelmate-ai")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR / "services" / "ai-service")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[training]"], check=True)

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Hãy chọn Runtime > Change runtime type > GPU trong Colab.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
SOURCE_DATASET = Path("training/data/travelmate_train.sample.jsonl")
PROCESSED_DIR = Path("training/data/processed")
subprocess.run(
    [
        sys.executable,
        "-m",
        "training.prepare_dataset",
        str(SOURCE_DATASET),
        "--output-dir",
        str(PROCESSED_DIR),
        "--seed",
        "42",
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "training.train_qlora",
        "--train-dataset",
        str(PROCESSED_DIR / "train.jsonl"),
        "--eval-dataset",
        str(PROCESSED_DIR / "validation.jsonl"),
        "--output-dir",
        "/content/travelmate-dry-run",
        "--dry-run",
    ],
    check=True,
)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

OUTPUT_DIR = Path("/content/drive/MyDrive/TravelMate/artifacts/travelmate-qwen3-4b-lora")
RUN_TRAINING = False  # Đổi thành True sau khi dry-run thành công.
EPOCHS = 1  # Dùng 1 cho smoke test; dùng 2-3 khi dataset chính thức đã đủ lớn.

if len((PROCESSED_DIR / "train.jsonl").read_text(encoding="utf-8").splitlines()) < 1000:
    print("CẢNH BÁO: Dataset hiện dưới 1.000 mẫu; adapter chỉ có giá trị kiểm tra pipeline.")

if RUN_TRAINING:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "training.train_qlora",
            "--train-dataset",
            str(PROCESSED_DIR / "train.jsonl"),
            "--eval-dataset",
            str(PROCESSED_DIR / "validation.jsonl"),
            "--output-dir",
            str(OUTPUT_DIR),
            "--epochs",
            str(EPOCHS),
        ],
        check=True,
    )
else:
    print("Chưa train. Đặt RUN_TRAINING=True khi bạn sẵn sàng dùng GPU.")